# Week 4

### Goals 
- Understand the agent cycle: decide, act, observe and respond.
- Make the boundary between LLM decisions and Python execution visible.

### Tasks 
- Draw the flow from a user request to tool selection, tool result and final answer.
- Implement a small manual loop with a fixed set of allowed actions.
- Log each action and observation in a readable trace.
- Limit the number of steps and end “gracefully” if the model cannot decide.
- Compare a direct LLM answer with a response grounded in the catalog.

### Expected Output 
- 04_agent_loop.ipynb with a trace for at least three requests.

---
#### Flow from user request to tool selection, tool result and final answer

<img src="../agent_cycle.png" width="70%">

---
#### Small loop with fixed set of allowed actions

In [3]:
from ollama import chat

# Simple mock function to simulate catalog lookup
def mock_catalog_lookup(genre: str) -> str:
    """Look up movies in the catalog by genre.
    Args:
        genre: The movie genre to search for (e.g., 'Action', 'Sci-Fi')
    """
    if genre.lower() == 'action':
        return "Found: The Matrix (1999), Gladiator (2000)"
    elif genre.lower() == 'sci-fi':
        return "Found: Interstellar (2014), Blade Runner 2049 (2017)"
    elif genre.lower() == 'fantasy':
        return "Found: The Lord of the Rings: The Fellowship of the Ring (2001), Harry Potter and the Sorcerer's Stone (2001)"
    return "No movies found for that genre."

available_functions = {'mock_catalog_lookup': mock_catalog_lookup}


# Function to run the agent with tracing
def run_agent_trace(user_prompt):
    messages = [{'role': 'user', 'content': user_prompt}]
    step_count = 0
    max_steps = 3

    print(f"\n=== TRACE FOR: '{user_prompt}' ===")
    
    while step_count < max_steps:
        step_count += 1
        print(f"\n--- Step {step_count} ---")

        response = chat(
            model='llama3.2',
            messages=messages,
            tools=[mock_catalog_lookup]
        )
        messages.append(response.message)

        # Check if the LLM wants to use a tool 
        if response.message.tool_calls:
            for tc in response.message.tool_calls:
                if tc.function.name in available_functions:
                    print(f"Action: Calling {tc.function.name} with {tc.function.arguments}")

                    # 'Act' Phase
                    result = available_functions[tc.function.name](**tc.function.arguments)

                    # 'Observe' Phase
                    print(f"Observation: {result}")
                    messages.append({'role': 'tool', 'tool_name': tc.function.name, 'content': str(result)})
        else:
            # 'Respond' Phase 
            print(f"Final Response: {response.message.content}")
            break

    # Graceful exit if the loop maxes out
    if step_count >= max_steps:
        print("\n[System] Step limit reached. Ending gracefully.")

**Run the agent with tracing for different user prompts:**

In [ ]:
run_agent_trace("Can you find me some Action movies?")


=== TRACE FOR: 'Can you find me some Action movies?' ===

--- Step 1 ---
Action: Calling mock_catalog_lookup with {'genre': 'Action'}
Observation: Found: The Matrix (1999), Gladiator (2000)

--- Step 2 ---
Final Response: Here are some action movie recommendations:

1. **The Matrix** (1999) - a groundbreaking sci-fi action film that explores the idea of a simulated reality.
2. **Gladiator** (2000) - an epic historical action film that won several Academy Awards, including Best Picture and Best Actor for Russell Crowe.

Would you like more recommendations or specific action movie suggestions?


In [5]:
run_agent_trace("What fantasy movies would you recommend?")


=== TRACE FOR: 'What fantasy movies would you recommend?' ===

--- Step 1 ---
Action: Calling mock_catalog_lookup with {'genre': 'Fantasy'}
Observation: Found: The Lord of the Rings: The Fellowship of the Ring (2001), Harry Potter and the Sorcerer's Stone (2001)

--- Step 2 ---
Final Response: Based on the tool call, I've found two fantasy movies that I'd like to recommend to you. Here are the details:

1. **The Lord of the Rings: The Fellowship of the Ring (2001)**: Directed by Peter Jackson, this film is the first installment of the iconic Lord of the Rings trilogy. It follows the journey of Frodo Baggins as he sets out to destroy the One Ring and save Middle-earth from the dark lord Sauron.
2. **Harry Potter and the Sorcerer's Stone (2001)**: Directed by Chris Columbus, this film is the first movie in the beloved Harry Potter series. It introduces us to the magical world of Hogwarts School of Witchcraft and Wizardry, where Harry Potter, a young orphan, discovers his true identity a

In [6]:
run_agent_trace("Do you have any Science Fiction movies?")


=== TRACE FOR: 'Do you have any Science Fiction movies?' ===

--- Step 1 ---
Action: Calling mock_catalog_lookup with {'genre': 'Sci-Fi'}
Observation: Found: Interstellar (2014), Blade Runner 2049 (2017)

--- Step 2 ---
Final Response: The science fiction movie "Interstellar" (2014) is directed by Christopher Nolan and stars Matthew McConaughey, Anne Hathaway, and Jessica Chastain. The film explores the possibilities of wormhole travel and the search for a new habitable planet for humanity.

The science fiction movie "Blade Runner 2049" (2017) is directed by Denis Villeneuve and stars Ryan Gosling, Harrison Ford, and Ana de Armas. The film is a sequel to the 1982 original "Blade Runner" and continues the story of a replicant hunter in a dystopian future.

Both films are thought-provoking and visually stunning, exploring complex themes such as humanity, artificial intelligence, and what it means to be alive.


In [7]:
run_agent_trace("I want a Romance movie.")


=== TRACE FOR: 'I want a Romance movie.' ===

--- Step 1 ---
Action: Calling mock_catalog_lookup with {'genre': 'Romance'}
Observation: No movies found for that genre.

--- Step 2 ---
Final Response: Based on your request for a Romance movie, I recommend "The Notebook" (2004) starring Ryan Gosling and Rachel McAdams. The movie follows the love story of two young souls, Noah and Allie, who are separated by social class but find their way back to each other years later.

Alternatively, you might enjoy "La La Land" (2016) directed by Damien Chazelle, starring Ryan Gosling and Emma Stone, which is a modern romantic musical about two aspiring artists who fall in love while chasing their dreams.

If you're in the mood for something more classic, "Roman Holiday" (1953) starring Audrey Hepburn and Gregory Peck could be a great choice. This timeless romantic comedy follows a royal princess who escapes her royal duties for a day of adventure and romance in Rome.

Which of these movies sounds ap

**Same user requests with simple llm call without tools:**

In [8]:
from ollama import ChatResponse

def simple_chat(user_prompt):
    response: ChatResponse = chat(
        model='llama3.2',
        messages=[
            {
                'role': 'user',
                'content': user_prompt,
            },
        ],
    )
    print(response.message.content)

In [9]:
simple_chat("Can you find me some Action movies?")

Here are some action movie recommendations:

**Recent Releases**

1. **Mission: Impossible - Fallout** (2018) - Tom Cruise stars as Ethan Hunt in this high-octane spy thriller.
2. **John Wick: Chapter 3 – Parabellum** (2019) - Keanu Reeves returns as the legendary hitman John Wick in this adrenaline-fueled sequel.
3. **Avengers: Endgame** (2019) - The epic conclusion to the Marvel Cinematic Universe's Infinity Saga, featuring an all-star cast.
4. **Extraction** (2020) - Chris Hemsworth stars as a black-market mercenary in this intense action-thriller.
5. **The Old Guard** (2020) - A Netflix original film starring Charlize Theron and Matthias Schoenaerts as immortal mercenaries.

**Classic Action Movies**

1. **Die Hard** (1988) - Bruce Willis stars as the iconic detective John McClane in this action-packed classic.
2. **Terminator 2: Judgment Day** (1991) - Arnold Schwarzenegger returns as the Terminator in this sci-fi action classic.
3. **Predator** (1987) - Arnold Schwarzenegger lead

In [10]:
simple_chat("What fantasy movies would you recommend?")

Here are some fantasy movies that I highly recommend:

**Epic Fantasy**

1. The Lord of the Rings: The Fellowship of the Ring (2001) - A classic adaptation of J.R.R. Tolkien's novel, set in Middle-earth.
2. The Princess Bride (1987) - A swashbuckling adventure with a heartwarming love story, memorable characters, and witty humor.
3. Pan's Labyrinth (2006) - A dark, imaginative, and visually stunning Spanish fantasy film set in post-Civil War Spain.
4. The Golden Compass (2007) - A visually breathtaking adaptation of Philip Pullman's novel, set in a parallel universe.
5. Harry Potter and the Philosopher's Stone (2001) - The first film in the beloved Harry Potter series, introducing us to the magical world of Hogwarts.

**Fantasy Adventure**

1. The Chronicles of Narnia: The Lion, the Witch and the Wardrobe (2005) - A magical adaptation of C.S. Lewis's classic novel, set in a magical land.
2. The Wizard of Oz (1939) - A classic musical fantasy film that has become an iconic part of pop c

In [11]:
simple_chat("Do you have any Science Fiction movies?")

I can definitely provide some science fiction movie recommendations. Here are a few:

**Classic Sci-Fi**

1. Blade Runner (1982) - a neo-noir classic set in a dystopian future.
2. The Matrix (1999) - a thought-provoking film that explores the nature of reality.
3. Star Wars: Episode IV - A New Hope (1977) - a space opera that launched the iconic franchise.
4. 2001: A Space Odyssey (1968) - a groundbreaking film that explores human evolution and technology.
5. Alien (1979) - a horror-sci-fi classic that spawned a successful franchise.

**Modern Sci-Fi**

1. Inception (2010) - a mind-bending film that explores the concept of shared dreaming.
2. Interstellar (2014) - a visually stunning film that explores wormholes and the search for a new home for humanity.
3. Her (2013) - a thought-provoking film that explores the consequences of artificial intelligence.
4. Arrival (2016) - a film that explores the concept of language and communication with alien life forms.
5. Ex Machina (2014) - a psy

In [12]:
simple_chat("I want a Romance movie.")

I'd be happy to help you with some romance movie recommendations. Here are a few suggestions:

**Classic Romances**

1. "The Notebook" (2004) - A timeless love story about two young souls who fall in love in the 1940s.
2. "Titanic" (1997) - A epic romance about a poor artist and a high-society woman who fall in love aboard the ill-fated ship.
3. "Casablanca" (1942) - A classic wartime romance set in Morocco, about a man who must choose between his love for his wife and his loyalty to his country.

**Modern Romances**

1. "Crazy, Stupid, Love" (2011) - A funny and heartwarming story about a man who's recently divorced and learns how to navigate the dating world with the help of a smooth-talking bachelor.
2. "The Proposal" (2009) - A witty and charming movie about a demanding boss who forces her assistant to pretend to be her fiancé, only to find themselves falling in love.
3. "To All the Boys I've Loved Before" (2018) - A sweet and lighthearted Netflix original about a high school girl 

---
### Observations

Comparing the direct LLM answer with the one using the catalog tool, it is visible that using the tool heavily influences the response. The LLM correctly understands the genre the user is looking for, extracts it in the right way from the text and uses it as input for the python function. The simply predefined movies that should be returned for each of the genres are returned to the LLM and it lists these movies as the recommendations, adding some additional information about the respective movies. The LLM also nicely transformed the written out genre 'Science Fiction' into sci-fi, although this may also have been luck and of course in a real application such small possible discrepancies need to be adressed. The request for a romance movie breaks the performance: Although the tool (python function) returns the string 'No movies found for that genre.', the final LLM answer still recommends/invents multiple movies instead of responding with a message telling the user that no such movies were found. Therefore for the real use case some clear guardrails will need to be implemented (e.g. combination with JSON format and clear system prompt like in week 2) to ensure no such hallucinations occur. In contrast, the simple request directly to the llm without any tool calls just gives a list of movies back, same as if I asked ChatGPT or similar AIs in the web. So those are most likely the movies of that category LLM has 'seen' the most during training and are not based on any catalog.

Also noticeable: The requests WITH a tool call were roughly twice as fast as the simple requests without any tools with the small llama3.2 model. This is most likely because the tool call to this small and simple python function is extremly fast and returns a short and simple answer. So the LLM itself has to do less work which speeds up the process.